# Critical Input DEQN: Relief Crowding-Out Tests

This notebook runs the marginal test we actually need after the baseline/no-relief anchors: keep the repair-active calibration fixed and vary only the external-relief channel. The target question is whether faster or stronger external relief lowers scarcity rents, repair value, and internal adaptation.

In [ ]:
from pathlib import Path
import sys
import torch

ROOT = Path('/content/econml')
if not (ROOT / 'src').exists():
    ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / 'notebooks') not in sys.path:
    sys.path.insert(0, str(ROOT / 'notebooks'))

from critical_input_sensitivity_helpers import train_rule_variant

ARTIFACT_ROOT = ROOT / 'baseline_artifacts' / 'critical_input_deqn'
OUT_ROOT = ARTIFACT_ROOT / 'relief_crowding_out_tests'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Keep the repair-active calibration fixed; vary only relief.
BASE_REPAIR_ACTIVE = {
    'repair_cost_share_10pct': 0.015,
    'mark_D': 0.35,
    'nu_qD': 1.25,
}

RELIEF_VARIANTS = {
    'baseline_relief': {
        'nu_pX': 0.05,
        'nu_qX': 0.60,
        'beta_X': 0.05,
        'mark_X': 0.15,
    },
    'strong_relief': {
        'nu_pX': 0.10,
        'nu_qX': 1.00,
        'beta_X': 0.10,
        'mark_X': 0.25,
    },
}

# The no-relief anchor is intentionally not retrained by default. It is loaded in the next cell
# from fixed_taylor_interior_repair_probe / bottleneck_interior_repair_probe if those folders exist.
TRAIN_NO_RELIEF_IF_ANCHOR_MISSING = False
NO_RELIEF = {
    'nu_pX': 0.0,
    'nu_qX': 0.0,
    'beta_X': 0.0,
    'mark_X': 0.0,
    'log_bar_lambda_X': -30.0,
}

POLICIES = ['fixed', 'bottleneck']
RUN_TRAIN = True
RETRAIN = False

# Light but credible rule-policy settings. Raise RULE_STEPS/QMC_* for final long runs.
TRAINING = dict(
    rule_steps=2500,
    qmc_train=128,
    qmc_val=256,
    n_val_states=512,
    stop_val_states=256,
    batch_size=1024,
    sim_batch_size=256,
    dtype='float64',
    natural_oracle_nodes=16,
    checkpoint_every=500,
    checkpoint_keep=6,
    scenario_q_weight=15.0,
    calm_anchor_weight=2.0,
    calm_residual_weight=2.0,
)

print('ROOT:', ROOT)
print('OUT_ROOT:', OUT_ROOT)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

if TRAIN_NO_RELIEF_IF_ANCHOR_MISSING:
    train_rule_variant(
        root=ROOT,
        output_dir=OUT_ROOT / 'no_relief',
        params={**BASE_REPAIR_ACTIVE, **NO_RELIEF},
        policies=POLICIES,
        run_train=RUN_TRAIN,
        retrain=RETRAIN,
        **TRAINING,
    )

for name, overrides in RELIEF_VARIANTS.items():
    train_rule_variant(
        root=ROOT,
        output_dir=OUT_ROOT / name,
        params={**BASE_REPAIR_ACTIVE, **overrides},
        policies=POLICIES,
        run_train=RUN_TRAIN,
        retrain=RETRAIN,
        **TRAINING,
    )


In [ ]:
from IPython.display import display
from critical_input_sensitivity_helpers import collect_rule_ir_diagnostics, plot_run_comparison, zip_folder

REPORT_DIR = OUT_ROOT / 'report'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

NO_RELIEF_ANCHORS = {
    'fixed': ARTIFACT_ROOT / 'fixed_taylor_interior_repair_probe',
    'bottleneck': ARTIFACT_ROOT / 'bottleneck_interior_repair_probe',
}

runs = []
for policy, folder in NO_RELIEF_ANCHORS.items():
    if (folder / 'run_config.json').exists() and ((folder / 'checkpoints' / f'{policy}_best.pt').exists() or (folder / f'{policy}.pt').exists()):
        runs.append({'variant': 'no_relief_anchor', 'policy': policy, 'output_dir': folder})
    else:
        print('Skipping missing no-relief anchor:', policy, folder)

for variant in RELIEF_VARIANTS:
    for policy in POLICIES:
        runs.append({'variant': variant, 'policy': policy, 'output_dir': OUT_ROOT / variant})

detail, summary, series_by_run = collect_rule_ir_diagnostics(
    artifact_root=ARTIFACT_ROOT,
    runs=runs,
    report_dir=REPORT_DIR,
    dtype=TRAINING['dtype'],
    burnin=80,
    horizon=100,
    presteps=5,
    relief_lag=8,
    natural_oracle_nodes=16,
    plot_each=False,
)

print('\nKEY SCENARIO SUMMARY')
display(summary)

for scenario in ['D_1x', 'D_3x', 'D_3x_X_lag']:
    plot_run_comparison(
        series_by_run,
        report_dir=REPORT_DIR,
        scenario=scenario,
        filename=f'{scenario}_relief_crowding_out_comparison.png',
    )

ZIP_PATH = Path('/content/relief_crowding_out_tests.zip')
zip_folder(OUT_ROOT, ZIP_PATH)

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except Exception as exc:
    print('Download helper skipped:', exc)
